# Encoder-side task-token allocation: every result of the paper

This notebook reproduces all results of the paper, end to end, **on Google Colab or locally** (VS Code / Jupyter).

**Setting.** One financial sentence, three tasks (Sentiment, FLS, ESG). Each task owns learnable task tokens;
one task token is sent as 4 complex symbols. **One channel** is used for both training and testing, chosen by
`CHANNEL` in the settings: **AWGN** (h = 1, the default) or **Rayleigh** (random fading on every symbol). Nothing is
mixed: each channel is a separate study with its own results folder. The allocator has **no channel-type input**:
it sees the sentence summary (CLS), the SNR reported by the receiver (CQI) and the operator priority. A total cap K
limits the task tokens per sentence, and a price λ per task token lets the allocator send fewer tokens than the cap.

**What the notebook produces**

**Part A: small token budget (4 task tokens per task, per-task maximum 4)**

| Section | Result |
|---|---|
| A1 | No-allocator saturation: accuracy with 1 to 4 tokens per task |
| A2 | Fixed cap 8 (λ = 0): allocator (gain form and priority embedding) vs equal, proportional and random splits and the hindsight oracle |
| A3 | Variable rate at cap 12: λ sweep (one allocator per λ), symbols needed for a target accuracy, per-SNR behaviour |
| A4 | Priority sweep at 0 dB (gain vs embedding) and one model vs a model trained at a fixed priority |
| A5 | Optional: allocator trained jointly with the pipeline (the approach we dropped) |

**Part B: large token budget (32 task tokens per task, per-task maximum K/2)**

| Section | Result |
|---|---|
| 1 | Data summary and the no-allocator saturation curve (why the number of tokens must adapt to SNR) |
| 2 | Choice of the price λ on the **validation** set (fewest symbols within ε of the equal split) |
| 3 | **The three benchmarks** at cap 16: no allocator with no cap, no allocator with cap 16, allocator with cap 16 |
| 4 | Accuracy versus bandwidth for every λ |
| 5 | Caps 8, 16, 32 and 64 |
| 6 | Ablation of every allocator input |
| 7 | Priority steering: our gain form versus a priority embedding |
| 8 | Summary of the statistically significant results |

All comparisons use the same frozen pipeline, the same 966 test sentences and the same channel noise, with
paired-bootstrap 95% confidence intervals (10,000 resamples). Every run is **resumable**: finished seeds are skipped,
so after a Colab disconnect simply run the notebook again.

**Run time (full, 3 seeds, one channel).** Part A about 2 hours and Part B about 4 to 6 hours on a
Colab T4; about half of that on an RTX-class laptop GPU.
`FAST = True` (1 seed, fewer runs) takes about a third of that.

In [ ]:
import os, sys, subprocess, shutil

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/karlkhoury/VIP.git"
BRANCH = "prior-paper-work"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/VIP_paper"          # dataset + results live here (survive disconnects)
    os.makedirs(DRIVE, exist_ok=True)
    REPO = "/content/VIP"
    CODE_ZIP = f"{DRIVE}/VIP_code.zip"
    if os.path.exists(CODE_ZIP):                         # option 1: code uploaded to Drive as a zip
        shutil.rmtree(REPO, ignore_errors=True)
        shutil.unpack_archive(CODE_ZIP, REPO)
        print("code unpacked from", CODE_ZIP)
    else:                                                # option 2: clone from GitHub
        url = REPO_URL
        try:                                             # only needed if the repository is private:
            from google.colab import userdata            # add GITHUB_TOKEN in the key icon panel
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass                                         # public repository: no token needed
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
        else:
            subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
        print("code cloned from GitHub branch", BRANCH)
    OUT_ROOT = os.environ.get("PAPER_OUT_ROOT", f"{DRIVE}/runs")
else:
    REPO = os.path.abspath(".")                         # run the notebook from the repository root
    assert os.path.exists(os.path.join(REPO, "run.py")), "open this notebook from the VIP repository root"
    OUT_ROOT = os.environ.get("PAPER_OUT_ROOT", os.path.join(REPO, "runs", "paper"))

os.chdir(REPO)
sys.path.insert(0, REPO)
print("running on", "Colab" if IN_COLAB else "local machine", "| repo:", REPO, "| results:", OUT_ROOT)

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "pyarrow", "huggingface_hub"], check=True)
import torch, transformers, pandas
print("torch", torch.__version__, "| transformers", transformers.__version__, "| pandas", pandas.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only: use SMOKE = True)")

### Settings
* `FAST = True`: one seed and a shorter list of runs, for a first look.
* `SMOKE = True`: a tiny random model on synthetic data; checks that every cell runs, in a few minutes on CPU.
  The numbers it produces are meaningless.
* `CHANNEL`: `"AWGN"` or `"Rayleigh"`. Each channel is a separate study with its own results folder
  (`runs` for AWGN, `runs_rayleigh` for Rayleigh), so the two never mix.
* `STAGE2`: how the allocators (stage 2) are trained. `"v1"` = 30 epochs, each validation sentence
  scored at one random SNR (the first runs). `"v2"` = 80 epochs, each validation sentence scored at
  5 random SNRs, so the kept epoch is not picked by noise. `"v2"` runs get their own names (`..._v2`):
  the stage-1 pipelines are reused and the `"v1"` results are never overwritten.
* `RUN_JOINT`: also run the optional single-stage (joint) training of Part A (about 30 minutes per seed).
* `EPSILON`: the largest accuracy loss (points) accepted when choosing λ (Section 2).

In [ ]:
FAST = False
SMOKE = False
EPSILON = 0.25
CHANNEL = "Rayleigh"      # "AWGN" or "Rayleigh"
RUN_JOINT = False
STAGE2 = "v2"             # "v1" (first runs) or "v2" (longer, less noisy allocator training)

from reproduce.runner import ABLATION, CAP8_VARIANTS, SmallStudy, Study
from reproduce.report import Report, SmallReport

S2 = {"v1": {}, "v2": {"tag": "_v2", "alloc_epochs": 80, "val_draws": 5}}[STAGE2]
RESULTS = OUT_ROOT if CHANNEL == "AWGN" else OUT_ROOT.rstrip("/\\") + "_" + CHANNEL.lower()

if SMOKE:
    study = Study(out_root=RESULTS + "_smoke", channel=CHANNEL, seeds=[0], tokens_per_task=4, caps=(4, 6), main_cap=6,
                  lambda_grid=(0.003, 0.01), epsilon=EPSILON, snrs=(-5, 5), priority_snrs=(0,),
                  priority_grid=(0.2, 0.8), ablation=("full", "snr_prio", "cls_snr"), smoke=True, **S2)
elif FAST:
    study = Study(out_root=RESULTS, channel=CHANNEL, seeds=[0], lambda_grid=(0.002, 0.003, 0.005, 0.01), epsilon=EPSILON,
                  ablation=("full", "snr_prio", "cls_snr", "cls_prio", "none", "full_conf"), **S2)
else:
    study = Study(out_root=RESULTS, channel=CHANNEL, seeds=[0, 1, 2], lambda_grid=(0.001, 0.002, 0.003, 0.005, 0.01, 0.02),
                  epsilon=EPSILON, **S2)
report = Report(study)

# Part A: 4 task tokens per task, same channel, same results folder, same seeds and SNRs
small_opts = ({"rate_lambdas": (0.01, 0.1), "priority_grid": (0.2, 0.8)} if SMOKE else
              {"rate_lambdas": (0.003, 0.01, 0.03)} if FAST else {})
small = SmallStudy(out_root=study.out_root, channel=CHANNEL, seeds=study.seeds, snrs=study.snrs,
                   smoke=SMOKE, **small_opts, **S2)
small_report = SmallReport(small)
print("channel", CHANNEL, "| stage 2:", STAGE2, "| results:", RESULTS, "| seeds", study.seeds, "| tokens per task", study.J, "| caps", study.caps, "| λ grid", study.lambda_grid)

### Data
The cleaned Financial PhraseBank file (4,825 sentences) is `data/phrasebank_multitask.parquet`.
On Colab, put a copy at `MyDrive/VIP_paper/phrasebank_multitask.parquet` to use exactly the same file;
otherwise it is rebuilt from the public sources (`common.prepare_data`, about 10 minutes, downloads about 900 MB).

In [ ]:
DATA = "data/phrasebank_multitask.parquet"
if not SMOKE and not os.path.exists(DATA):
    os.makedirs("data", exist_ok=True)
    if IN_COLAB and os.path.exists(f"{DRIVE}/phrasebank_multitask.parquet"):
        shutil.copy(f"{DRIVE}/phrasebank_multitask.parquet", DATA)
        print("copied the dataset from Drive")
    else:
        subprocess.run([sys.executable, "-m", "common.prepare_data", "--out", DATA], check=True)
print("dataset ready" if (SMOKE or os.path.exists(DATA)) else "dataset missing")

In [ ]:
# unit tests of the physical layer, the allocator and the baselines (CPU, about 1 minute)
r = subprocess.run([sys.executable, "-m", "tests.test_all"], capture_output=True, text=True,
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))
print("\n".join(l for l in r.stdout.splitlines() if l.startswith(("PASS", "FAIL"))) or r.stdout[-2000:] + r.stderr[-2000:])

# Part A. Small token budget: 4 task tokens per task
Each task owns 4 task tokens (12 in total, at most 48 symbols); a task may send 1 to 4 of them. Stage 1 trains the
pipeline on the chosen channel only; stage 2 trains each allocator alone on the frozen pipeline. Every method below
is evaluated on that same frozen pipeline, with the same test sentences and the same fixed fading and noise.

## A1. Stage 1 and the saturation curve
FinBERT, the 12 task tokens, the per-task channel encoders and decoders, the pooling layers and the heads are trained
for 12 epochs with nested dropout. The test evaluation then sends k = 1 to 4 tokens per task, with no allocator.

In [ ]:
small.run(small.pipeline_config())
small_report.dataset()
small_report.saturation()

## A2. Fixed cap 8 (λ = 0): splitting eight tokens
Every method sends about 8 tokens (32 symbols); only the split between the tasks differs. Two allocators are
trained: the standard one (priority enters only the final score, through the gain) and one with the
priority embedded as an input of the hidden layer.
The hindsight oracle uses the test labels and the realized noise, so it is an upper bound, not a method.

In [ ]:
for variant in CAP8_VARIANTS:
    small.run(small.cap8_config(variant))
small_report.fixed_cap()

## A3. Variable rate at cap 12
The cap (12 tokens) no longer limits the allocator, so the price λ decides how many tokens it sends. Each λ is
compared with fixed splits (no allocator).

In [ ]:
for lam in small.lambda_grid:
    small.run(small.rate_config(lam))
small_report.rate()

## A4. Priority steering at 0 dB
The cap-8 allocators are re-evaluated (not retrained) while the ESG priority goes from 0.1 to 0.9. A separate
allocator is trained (and validated) only at w = (0.1, 0.1, 0.8) to check that one model trained with random
priorities serves every priority.

In [ ]:
small.run(small.priority_config("gain"))
small.run(small.priority_config("embed"))
small.run(small.fixed_priority_config())
small_report.priority()

## A5. Optional: joint (single-stage) training
The allocator trained together with the pipeline, as first designed. Set `RUN_JOINT = True` to run it.

In [ ]:
if RUN_JOINT:
    small.run(small.joint_config())
small_report.joint()

## Part A summary

In [ ]:
small_report.summary()

# Part B. Large token budget: 32 task tokens per task
Each task owns 32 task tokens (96 in total) and may send at most K/2 under a cap K. Same channel as Part A.

## 1. Stage 1: the pipeline, and the no-allocator saturation curve
FinBERT, the 96 task tokens, the per-task channel encoders and decoders, the pooling layers and the heads are trained
for 12 epochs on the chosen channel with nested dropout (each task keeps a random number of its first tokens). This model is then
frozen and shared by every experiment below. Its test evaluation sends k tokens per task for k = 1 to 32.

In [ ]:
study.run(study.pipeline_config())
report.saturation()

## 2. Stage 2 and the choice of λ
For each price λ in the grid, the allocator alone is trained on the frozen pipeline at cap 16 (per-task max 8) and
evaluated on the validation and test sets.

**Selection rule (validation only).** Among the prices whose validation accuracy is within ε points of the
no-allocator equal split at the same cap, choose the one that sends the fewest symbols.

This is the ε-constraint form of our problem, *minimize the average number of symbols subject to an accuracy
no lower than the reference minus ε*. Its Lagrangian is the training loss (cross-entropy plus λ times the tokens),
so λ is the Lagrange multiplier of the accuracy constraint: a larger λ means fewer symbols and lower accuracy.
The grid search finds the multiplier that makes the constraint active on held-out data. The test set then
confirms the choice with a **non-inferiority** test: the lower end of the 95% confidence interval of the accuracy
difference must lie above −ε, and the symbol reduction must be significant.

In [ ]:
for lam in study.lambda_grid:
    study.run(study.lambda_config(lam))
LAM = report.select_lambda()

## 3. The three benchmarks (test set)
1. **No allocator, no cap**: every task sends all 32 tokens (384 symbols).
2. **No allocator, cap 16**: equal split 6/5/5 (64 symbols).
3. **Learned allocator, cap 16** at the selected λ.

In [ ]:
report.benchmarks()

## 4. Accuracy versus bandwidth for every λ (test)

In [ ]:
report.tradeoff()

## 5. Caps 8, 16, 32 and 64
Same pipeline and selected λ; per-task maximum = cap / 2.

In [ ]:
for cap in study.caps:
    study.run(study.cap_config(cap, LAM))
report.caps()

## 6. Which allocator inputs matter
Allocators with different inputs (cap 16, selected λ), evaluated at three priorities: equal, ESG-heavy (0.1, 0.1, 0.8)
and Sentiment-heavy (0.8, 0.1, 0.1). "CLS + SNR + priority" is the selected allocator itself (re-evaluated, not
retrained). With a single channel, the channel type is constant, so it is not an input here.

In [ ]:
for variant in study.ablation:
    study.run(study.ablation_config(variant, LAM))
report.ablation()

## 7. Priority steering
The operator raises the ESG priority from 0.1 to 0.9. **gain** is our allocator
(score = base + softplus(gain) × priority, guaranteed never to give a task fewer tokens when its priority rises);
**embed** is the same allocator with the priority embedded in its hidden layer, as in the original design figure.

In [ ]:
study.run(study.priority_config("gain", LAM))
study.run(study.priority_config("embed", LAM))
report.priority()

## 8. Summary

In [ ]:
report.summary()